# Week 9 — Multiple Linear Regression

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week09/`)
- `biz_customers.csv` — ลูกค้า 1203 คน
- `biz_campaigns.csv` — แคมเปญ 240 รายการ

## เครื่องมือของสัปดาห์นี้: statsmodels

ทุกข้อใช้ OLS ผ่าน **สูตร (formula)** ตามที่โจทย์กำหนด:
```python
import statsmodels.formula.api as smf
model = smf.ols('y ~ x1 + x2 + x3', data=df).fit()
```
- โมเดลมี **intercept เสมอ** โดยอัตโนมัติ (`Intercept`)
- ชื่อสัมประสิทธิ์ใช้ `model.params['x1']`, t-stat ใช้ `model.tvalues['x1']`,
  p-value ใช้ `model.pvalues['x1']`, R² ใช้ `model.rsquared`, Adjusted R² ใช้ `model.rsquared_adj`
- **AIC/BIC ใช้ค่าที่ statsmodels รายงาน** (`model.aic`, `model.bic`)

## จุดที่คนตกมักที่สุด

1. **CI ของค่าเฉลี่ย ≠ PI ของรายบุคคล**
   - ค่าเฉลี่ย → `pred.conf_int(alpha=0.05)` (**แคบ**)
   - รายบุคคล → `pred.summary_frame(alpha=0.05)['obs_ci_upper']` (**กว้างกว่า** เพราะมีความแปรปรวนของแต่ละคน)
   - ใน statsmodels เวอร์ชันใหม่ `pred.obs_ci(...)` **ถูกเอาออกแล้ว** ให้ใช้ `summary_frame`
2. **Adjusted R² มีค่าต่ำกว่า R² เสมอ** เพราะลงโทษตัวแปรที่เพิ่มเข้ามา
3. **AIC ต่ำสุด = โมเดลดีสุด** (ทั้ง AIC และ BIC: ต่ำดี, และ BIC ลงโทษจำนวนพารามิเตอร์หนักกว่า)
4. **Backward elimination ต้อง fit ใหม่ทุกรอบ** และตัด "ตัวที่มี p-value สูงสุด" ทีละตัว
5. **ข้อ 9.6** ตัวแปร 5 ตัว → ชุดย่อยที่ไม่ว่างทั้งหมด = 2⁵ − 1 = **31 ชุด** ใช้ `itertools.combinations`
6. **แบ่ง train ตามตำแหน่งแถว** ไม่สุ่ม (ข้อ 9.4 ใช้ 960 แถวแรก)

> ทุก cell ข้อในไฟล์นี้**เป็นอิสระ** คัดลอกไปรันเดี่ยว ๆ ได้เลย
> (มี import และโหลดไฟล์ของตัวเองครบ) อ่านข้อไหนก็ดูข้อนั้นได้เลย

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week09')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

ไฟล์ครบแล้ว: data\week09


In [2]:
# --- ตรวจไฟล์ข้อมูล + เวอร์ชันเครื่องมือ (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd
import statsmodels, statsmodels.formula.api as smf

DATA_DIR = Path(f'data/week09')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv']

print('statsmodels', statsmodels.__version__, '| pandas', pd.__version__)
for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์')

statsmodels 0.15.0 | pandas 3.0.5
biz_campaigns.csv        240 แถว x  7 คอลัมน์
biz_customers.csv       1203 แถว x 13 คอลัมน์


## ข้อ 9.1 — สัมประสิทธิ์ของ loyalty_member ในโมเดล 5 ตัวแปร

- `loyalty_member` เป็นตัวแปร 0/1 จึงเป็น dummy variable ในตัว
- ตัวสัมประสิทธิ์คือ **ผลต่างของยอดซื้อเฉลี่ยระหว่างสมาชิกกับไม่สมาชิก**
  เมื่อคุมตัวแปรอื่นทั้ง 4 ตัวไว้แล้ว

In [3]:
# --- ข้อ 9.1 ---
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week09') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.ols('spend_k ~ income_k + tenure_months + email_opens + web_visits + loyalty_member',
            data=cust).fit()
answer = float(m.params['loyalty_member'])

print('n =', int(m.nobs))
print(m.summary().tables[1])
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

n = 1203
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          0.7184      0.162      4.448      0.000       0.402       1.035
income_k           0.0488      0.003     18.063      0.000       0.043       0.054
tenure_months      0.0047      0.002      2.386      0.017       0.001       0.009
email_opens        0.0083      0.019      0.441      0.659      -0.029       0.045
web_visits         0.1175      0.011     10.542      0.000       0.096       0.139
loyalty_member     0.3675      0.093      3.966      0.000       0.186       0.549
ค่าที่กรอก: 4 ตำแหน่ง -> 0.3675


## ข้อ 9.2 — Pearson r ระหว่าง income_k กับ basket_size

- ตรวจก่อนใส่ร่วมกันในโมเดล เพราะถ้าสองตัวแปรสัมพันธ์กันแรง จะเกิด **multicollinearity**
  ทำให้ทั้งสองสัมประสิทธิ์ประมาณผิดทาง (ดูข้อ 9.10)

In [4]:
# --- ข้อ 9.2 ---
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week09') / 'biz_customers.csv')   # ลูกค้า 1203 คน

answer = float(cust['income_k'].corr(cust['basket_size']))

print('r =', answer, '-> r^2 =', round(answer ** 2, 4), '= สัดส่วนความแปรปรวนที่ซ้อนกัน')
print('|r| > 0.7 ถือว่าสูงพอที่จะทำให้เกิด multicollinearity ในโมเดล')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

r = 0.3521921106516057 -> r^2 = 0.124 = สัดส่วนความแปรปรวนที่ซ้อนกัน
|r| > 0.7 ถือว่าสูงพอที่จะทำให้เกิด multicollinearity ในโมเดล
ค่าที่กรอก: 4 ตำแหน่ง -> 0.3522


## ข้อ 9.3 — t-statistic ของสัมประสิทธิ์ income_k

- ทดสอบ H₀: β = 0 ด้วย `model.tvalues['income_k']`
- เมื่อใส่ตัวแปรอื่นเข้าไปด้วย ความหมายของ β เปลี่ยนเป็น "อิทธิพลของรายได้ **เมื่อคุมตัวแปรอื่นคงที่**"

In [5]:
# --- ข้อ 9.3 ---
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week09') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.ols('spend_k ~ income_k + web_visits + loyalty_member + age', data=cust).fit()
answer = float(m.tvalues['income_k'])

print('n =', int(m.nobs), '| df_resid =', int(m.df_resid))
print('beta  =', round(float(m.params['income_k']), 6), '| SE =', round(float(m.bse['income_k']), 6))
print('t     =', round(answer, 4), '| p =', f'{float(m.pvalues["income_k"]):.4g}')
print('(t ขนาด 17.5 = มีนัยสำคัญมาก ทุกรายได้เพิ่มขึ้น 1 พันบาท ยอดซื้อเพิ่มขึ้นประมาณ',
      round(float(m.params['income_k']), 3), 'พันบาท เมื่อคุมตัวแปรอื่น)')
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

n = 1203 | df_resid = 1198
beta  = 0.048909 | SE = 0.002793
t     = 17.5137 | p = 2.537e-61
(t ขนาด 17.5 = มีนัยสำคัญมาก ทุกรายได้เพิ่มขึ้น 1 พันบาท ยอดซื้อเพิ่มขึ้นประมาณ 0.049 พันบาท เมื่อคุมตัวแปรอื่น)
ค่าที่กรอก: 2 ตำแหน่ง -> 17.51


## ข้อ 9.4 — Adjusted R² บนชุด train (960 แถว)

- ใช้ **960 แถวแรก** เท่านั้น
- Adjusted R² = 1 − (1 − R²)(n − 1)/(n − p − 1) จึง**มีค่าต่ำกว่า R² เสมอ**

In [6]:
# --- ข้อ 9.4 ---
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week09') / 'biz_customers.csv')   # ลูกค้า 1203 คน

train = cust.iloc[:960]                       # 960 แถวแรก (ไม่สุ่ม)
m = smf.ols('spend_k ~ tenure_months + loyalty_member + basket_size + email_opens',
            data=train).fit()
answer = float(m.rsquared_adj)

print('n =', int(m.nobs), '| จำนวนตัวแปร (ไม่รวม intercept) =', len(m.params) - 1)
print('R2 =', round(float(m.rsquared), 6), '-> Adjusted R2 =', round(answer, 6))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

n = 960 | จำนวนตัวแปร (ไม่รวม intercept) = 4
R2 = 0.083611 -> Adjusted R2 = 0.079772
ค่าที่กรอก: 4 ตำแหน่ง -> 0.0798


## ข้อ 9.5 — BIC ของโมเดลยอดขาย

- BIC ลงโทษจำนวนพารามิเตอร์มากกว่า AIC (ใช้ log n คูณจำนวนพารามิเตอร์)
- ใช้ค่าที่ statsmodels รายงาน: `model.bic`

In [7]:
# --- ข้อ 9.5 ---
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

camp = pd.read_csv(Path('data/week09') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

m = smf.ols('sales ~ tv + radio + social + email', data=camp).fit()
answer = float(m.bic)

print('n =', int(m.nobs), '| จำนวนพารามิเตอร์ทั้งหมด (รวม intercept) =', len(m.params))
print('AIC =', round(float(m.aic), 4), '| BIC =', answer)
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

n = 240 | จำนวนพารามิเตอร์ทั้งหมด (รวม intercept) = 5
AIC = 1037.6128 | BIC = 1055.0159694517995
ค่าที่กรอก: 2 ตำแหน่ง -> 1055.02


## ข้อ 9.6 — เลือกโมเดลจากทุกชุดย่อย (31 ชุด) ด้วย AIC

- ตัวแปร 5 ตัว → ชุดย่อยที่ไม่ว่าง = 2⁵ − 1 = **31 ชุด**
- ใช้ `itertools.combinations` วนทุกขนาด
- เลือกโมเดลที่ **AIC ต่ำที่สุด** แล้วรายงาน Adjusted R² ของโมเดลนั้น

In [8]:
# --- ข้อ 9.6 ---
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week09') / 'biz_customers.csv')   # ลูกค้า 1203 คน
import itertools

cands = ['income_k', 'tenure_months', 'loyalty_member', 'age', 'email_opens']
results = []
for k in range(1, len(cands) + 1):
    for combo in itertools.combinations(cands, k):
        fit = smf.ols('spend_k ~ ' + ' + '.join(combo), data=cust).fit()
        results.append((float(fit.aic), float(fit.rsquared_adj), combo))

results.sort(key=lambda r: r[0])              # AIC ต่ำสุดขึ้นก่อน
answer = results[0][1]

print('จำนวนชุดย่อยที่ทดลอง =', len(results), '(ควรเป็น 31)')
print('5 โมเดล AIC ต่ำสุด:')
for a, ar, combo in results[:5]:
    print(f'  AIC={a:9.4f}  AdjR2={ar:.6f}  {"+".join(combo)}')
print('โมเดลที่ชนะ =', '+'.join(results[0][2]), '| AIC =', round(results[0][0], 4))
print('ค่าที่กรอก: 4 ตำแหน่ง (Adjusted R2 ของโมเดลที่ชนะ) ->', f'{answer:.4f}')

จำนวนชุดย่อยที่ทดลอง = 31 (ควรเป็น 31)
5 โมเดล AIC ต่ำสุด:
  AIC=4167.3146  AdjR2=0.235468  income_k+tenure_months+loyalty_member+age
  AIC=4169.1842  AdjR2=0.234912  income_k+tenure_months+loyalty_member+age+email_opens
  AIC=4170.3650  AdjR2=0.232892  income_k+loyalty_member+age
  AIC=4172.2624  AdjR2=0.232317  income_k+loyalty_member+age+email_opens
  AIC=4178.0150  AdjR2=0.227998  income_k+tenure_months+loyalty_member
โมเดลที่ชนะ = income_k+tenure_months+loyalty_member+age | AIC = 4167.3146
ค่าที่กรอก: 4 ตำแหน่ง (Adjusted R2 ของโมเดลที่ชนะ) -> 0.2355


## ข้อ 9.7 — Backward elimination

ขั้นตอน (ทำซ้ำจนไม่มีตัวไหน p > 0.05):
1. fit โมเดลด้วยตัวแปรชุดปัจจุบัน
2. ดู `pvalues` ของตัวแปร (ไม่รวม Intercept)
3. ถ้ามีค่าสูงสุด > 0.05 → **ตัดตัวนั้นออก 1 ตัว** แล้วกลับไปข้อ 1 (fit ใหม่ทุกครั้ง!)
4. คำตอบคือ `aic` ของโมเดลสุดท้าย

In [9]:
# --- ข้อ 9.7 ---
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week09') / 'biz_customers.csv')   # ลูกค้า 1203 คน

current = ['income_k', 'loyalty_member', 'age', 'basket_size', 'email_opens', 'orders_90d']
round_no = 0
while True:
    m = smf.ols('spend_k ~ ' + ' + '.join(current), data=cust).fit()
    pv = m.pvalues.drop('Intercept')          # ไม่สนใจ p ของ intercept
    worst = pv.idxmax()
    round_no += 1
    print(f'รอบที่ {round_no}: AIC={float(m.aic):.4f} | max p = {float(pv.max()):.4f} ({worst})')
    print('        ตัวแปรที่เหลือ:', current)
    if float(pv.max()) > 0.05:
        current = [c for c in current if c != worst]
        print('        -> ตัด', worst, 'ออก แล้ว fit ใหม่')
    else:
        break

answer = float(m.aic)
print()
print('โมเดลสุดท้ายมีตัวแปร:', current)
print('ค่าที่กรอก: 2 ตำแหน่ง (AIC ของโมเดลสุดท้าย) ->', f'{answer:.2f}')

รอบที่ 1: AIC=4153.0870 | max p = 0.7068 (basket_size)
        ตัวแปรที่เหลือ: ['income_k', 'loyalty_member', 'age', 'basket_size', 'email_opens', 'orders_90d']
        -> ตัด basket_size ออก แล้ว fit ใหม่


รอบที่ 2: AIC=4151.2294 | max p = 0.6240 (email_opens)
        ตัวแปรที่เหลือ: ['income_k', 'loyalty_member', 'age', 'email_opens', 'orders_90d']
        -> ตัด email_opens ออก แล้ว fit ใหม่
รอบที่ 3: AIC=4149.4709 | max p = 0.0033 (age)
        ตัวแปรที่เหลือ: ['income_k', 'loyalty_member', 'age', 'orders_90d']

โมเดลสุดท้ายมีตัวแปร: ['income_k', 'loyalty_member', 'age', 'orders_90d']
ค่าที่กรอก: 2 ตำแหน่ง (AIC ของโมเดลสุดท้าย) -> 4149.47


## ข้อ 9.8 — ขอบล่างของ 95% CI ของ **ค่าเฉลี่ย**

- โจทย์ถามยอดซื้อ**เฉลี่ย**ของกลุ่ม → ใช้ `conf_int()` (**แคบ**)
  ไม่ใช่ PI ของรายบุคคล (ซึ่งกว้างกว่ามาก)
- ต้องสร้าง DataFrame ของจุดที่ถาม ให้**ชื่อคอลัมน์ตรงกับตอน fit**

In [10]:
# --- ข้อ 9.8 ---
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week09') / 'biz_customers.csv')   # ลูกค้า 1203 คน

m = smf.ols('spend_k ~ income_k + web_visits + tenure_months', data=cust).fit()

new = pd.DataFrame({'income_k': [25], 'web_visits': [9], 'tenure_months': [31]})
pred = m.get_prediction(new)
mean_val = float(np.asarray(pred.predicted_mean).ravel()[0])
ci = np.asarray(pred.conf_int(alpha=0.05)).ravel()
pi = np.asarray(pred.summary_frame(alpha=0.05)[[
    'obs_ci_lower', 'obs_ci_upper']].to_numpy()).ravel()
answer = float(ci[0])

print('จุดที่ถาม: income_k=25, web_visits=9, tenure_months=31')
print('ค่าทำนาย (ค่าเฉลี่ย) =', round(mean_val, 6))
print('95% CI ของค่าเฉลี่ย   = [', round(float(ci[0]), 4), ',', round(float(ci[1]), 4), ']  <- คำตอบ')
print('95% PI ของรายบุคคล   = [', round(float(pi[0]), 4), ',', round(float(pi[1]), 4), ']  <- กว้างกว่ามาก')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จุดที่ถาม: income_k=25, web_visits=9, tenure_months=31
ค่าทำนาย (ค่าเฉลี่ย) = 3.328874
95% CI ของค่าเฉลี่ย   = [ 3.2474 , 3.4104 ]  <- คำตอบ
95% PI ของรายบุคคล   = [ 0.7316 , 5.9261 ]  <- กว้างกว่ามาก
ค่าที่กรอก: 4 ตำแหน่ง -> 3.2474


## ข้อ 9.9 — ขอบบนของ 95% Prediction Interval

- โจทย์ถามยอดขายของ**แคมเปญหนึ่งรายการ** → ต้องใช้ **PI** ไม่ใช่ CI
- ใน statsmodels รุ่นใหม่ `pred.obs_ci(...)` ถูกเอาออกแล้ว
  ให้ใช้ `pred.summary_frame(alpha=0.05)` แล้วอ่านคอลัมน์ `obs_ci_upper`

In [11]:
# --- ข้อ 9.9 ---
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

camp = pd.read_csv(Path('data/week09') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

m = smf.ols('sales ~ tv + radio + social + email', data=camp).fit()

new = pd.DataFrame({'tv': [253], 'radio': [48], 'social': [26], 'email': [27]})
pred = m.get_prediction(new)
mean_val = float(np.asarray(pred.predicted_mean).ravel()[0])
ci = np.asarray(pred.conf_int(alpha=0.05)).ravel()          # ของค่าเฉลี่ย (แคบ)
sf = pred.summary_frame(alpha=0.05)                          # ของรายบุคคล (กว้าง)
pi_low = float(sf['obs_ci_lower'].iloc[0])
pi_high = float(sf['obs_ci_upper'].iloc[0])
answer = pi_high

print('แคมเปญใหม่: tv=253, radio=48, social=26, email=27')
print('ค่าทำนาย =', round(mean_val, 4))
print('95% CI ของค่าเฉลี่ย = [', round(float(ci[0]), 4), ',', round(float(ci[1]), 4), ']  (ไม่ใช่คำตอบ)')
print('95% PI ของรายบุคคล = [', round(pi_low, 4), ',', round(pi_high, 4), ']  <- คำตอบ')
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

แคมเปญใหม่: tv=253, radio=48, social=26, email=27
ค่าทำนาย = 34.5408
95% CI ของค่าเฉลี่ย = [ 33.8029 , 35.2787 ]  (ไม่ใช่คำตอบ)
95% PI ของรายบุคคล = [ 30.3767 , 38.7049 ]  <- คำตอบ
ค่าที่กรอก: 2 ตำแหน่ง -> 38.70


## ข้อ 9.10 — ผลของ multicollinearity: สัมประสิทธิ์เปลี่ยนไปแค่ไหน

- simple: `spend_k ~ basket_size`
- multiple: `spend_k ~ basket_size + income_k`
- คำตอบ = **สัมประสิทธิ์ของ `basket_size` ในโมเดล multiple ลบด้วยในโมเดล simple**
  (ตามโจทย์ให้ "ลบด้วย" = multiple − simple → ค่าติดลบ เพราะสัมประสิทธิ์หายไปเกือบทั้งหมด)

In [12]:
# --- ข้อ 9.10 ---
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

cust = pd.read_csv(Path('data/week09') / 'biz_customers.csv')   # ลูกค้า 1203 คน

simple = smf.ols('spend_k ~ basket_size', data=cust).fit()
multi = smf.ols('spend_k ~ basket_size + income_k', data=cust).fit()
b_simple = float(simple.params['basket_size'])
b_multi = float(multi.params['basket_size'])
answer = b_multi - b_simple

print('r(income_k, basket_size) =', round(float(cust['income_k'].corr(cust['basket_size'])), 4),
      '-> ตัวแปรทั้งสองอธิบายข้อมูลทับซ้อนกัน')
print('simple  : beta(basket_size) =', round(b_simple, 6), '| SE =', round(float(simple.bse['basket_size']), 6))
print('multiple: beta(basket_size) =', round(b_multi, 6), '| SE =', round(float(multi.bse['basket_size']), 6))
print('p-value ในโมเดล multiple =', f'{float(multi.pvalues["basket_size"]):.4f}', '(ไม่มีนัยสำคัญอีกต่อไป)')
print('ผลต่าง (multiple - simple) =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

r(income_k, basket_size) = 0.3522 -> ตัวแปรทั้งสองอธิบายข้อมูลทับซ้อนกัน
simple  : beta(basket_size) = 0.283679 | SE = 0.052817
multiple: beta(basket_size) = 3.5e-05 | SE = 0.051447
p-value ในโมเดล multiple = 0.9995 (ไม่มีนัยสำคัญอีกต่อไป)
ผลต่าง (multiple - simple) = -0.28364407174327305
ค่าที่กรอก: 4 ตำแหน่ง -> -0.2836


## สรุปคำตอบทั้ง 10 ข้อ (Week 9)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่องที่เว็บยอมรับ` (ไม่มีการผ่อนปรนเพิ่ม)

| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 9.1 | สัมประสิทธิ์ loyalty_member (โมเดล 5 ตัวแปร) | **0.3675** | 4 ตำแหน่ง | 0.00015 |
| 9.2 | Pearson r ระหว่าง income_k กับ basket_size | **0.3522** | 4 ตำแหน่ง | 0.00015 |
| 9.3 | t-statistic ของ income_k (คุม 3 ตัวแปร) | **17.51** | 2 ตำแหน่ง | 0.015 |
| 9.4 | Adjusted R² บน 960 แถวแรก | **0.0798** | 4 ตำแหน่ง | 0.00015 |
| 9.5 | BIC ของ sales ~ tv+radio+social+email | **1055.02** | 2 ตำแหน่ง | 0.015 |
| 9.6 | Adjusted R² ของโมเดลที่ AIC ต่ำสุด (31 ชุด) | **0.2355** | 4 ตำแหน่ง | 0.00015 |
| 9.7 | AIC หลัง backward elimination จบ | **4149.47** | 2 ตำแหน่ง | 0.015 |
| 9.8 | ขอบล่าง 95% CI ของค่าเฉลี่ยที่ (25, 9, 31) | **3.2474** | 4 ตำแหน่ง | 0.00015 |
| 9.9 | ขอบบน 95% PI ของ sales ที่ (253,48,26,27) | **38.70** | 2 ตำแหน่ง | 0.015 |
| 9.10 | beta ของ basket_size: multiple − simple | **-0.2836** | 4 ตำแหน่ง | 0.00015 |

### สิ่งที่ควรจำจากสัปดาห์นี้

- **9.6** โมเดลที่ AIC ต่ำสุดคือ `income_k + tenure_months + loyalty_member + age` (ตัด `email_opens` ทิ้ง)
- **9.7** backward elimination ตัด 3 ตัว: `basket_size` → `email_opens` → `age` เหลือ
  `income_k + loyalty_member + age + orders_90d`
- **9.8 vs 9.9** CI ของค่าเฉลี่ยกว้าง ~0.08 แต่ PI ของรายบุคคลกว้าง ~2.6 — **คนละแนวคิด**
- **9.10** สัมประสิทธิ์ `basket_size` ลดจาก 0.2837 เหลือ 0.00003 ทันทีที่ใส่ `income_k`
  เพราะสองตัวแปรนี้อธิบายยอดซื้อทับซ้อนกัน (r = 0.35) → อาการ **multicollinearity**

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน